# 06c — Deep Learning Comparison (Weekly Version)

**Goal:** decide which deep learning model advances to the final tournament in notebook 07.

**Methodology:** mirror the structure of 04c (statistical) and 05c (ML) for consistency.
Compare LSTM (06a) vs TFT (06b) on:
- Overall metrics (val + test)
- Per-item head-to-head wins
- Per-section breakdown
- Per-volume-tier breakdown
- Wilcoxon signed-rank test
- Strategic trade-offs (training time, complexity, interpretability)
- Final 3-criterion decision

**Bar to clear:** the ML champion's test WAPE. If neither LSTM nor TFT beats it, we'll
make a defensible argument about that — deep learning isn't required to win on a dataset
this size, and showing both gives the soutenance jury a complete picture.


## 1. Setup and load both models' outputs


In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

DATASETS_DIR = '../../datasets'

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.titleweight'] = 'bold'

lstm_pred = pd.read_csv(os.path.join(DATASETS_DIR, 'lstm_predictions_weekly.csv'),
                         parse_dates=['ds'])
tft_pred = pd.read_csv(os.path.join(DATASETS_DIR, 'tft_predictions_weekly.csv'),
                        parse_dates=['ds'])

lstm_metrics = pd.read_csv(os.path.join(DATASETS_DIR, 'lstm_metrics_weekly.csv'))
tft_metrics = pd.read_csv(os.path.join(DATASETS_DIR, 'tft_metrics_weekly.csv'))

with open(os.path.join(DATASETS_DIR, 'lstm_summary_weekly.json')) as f:
    lstm_summary = json.load(f)
with open(os.path.join(DATASETS_DIR, 'tft_summary_weekly.json')) as f:
    tft_summary = json.load(f)

# Load the ML champion (XGBoost) as the bar to clear
with open(os.path.join(DATASETS_DIR, 'lightgbm_summary_weekly.json')) as f:
    ml_champion = json.load(f)  # LightGBM is the ML champion (3-0 vs XGBoost)

print(f"LSTM: {len(lstm_pred):,} predictions, {lstm_metrics['item_name'].nunique()} items")
print(f"TFT:  {len(tft_pred):,} predictions, {tft_metrics['item_name'].nunique()} items")
print(f"ML champion bar: test WAPE {ml_champion['overall_metrics']['test']['wape']:.4f}")


## 2. Side-by-side overall metrics


In [ ]:
rows = []
for split in ['val', 'test']:
    for name, summary in [('LSTM', lstm_summary), ('TFT', tft_summary), ('XGBoost (ML champ)', ml_champion)]:
        m = summary['overall_metrics'][split]
        rows.append({
            'Model': name, 'Split': split,
            'MAE': m['mae'], 'RMSE': m['rmse'],
            'WAPE': m['wape'], 'sMAPE': m.get('smape', np.nan),
            'R²': m['r2'], 'Bias': m['bias'],
        })

overall_df = pd.DataFrame(rows)
print("=" * 80)
print("OVERALL METRICS — LSTM vs TFT (with XGBoost as reference)")
print("=" * 80)
print(overall_df.to_string(index=False, float_format='%.4f'))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
dl_only = overall_df[overall_df['Model'].isin(['LSTM', 'TFT'])]
for ax, metric in zip(axes, ['WAPE', 'R²', 'Bias']):
    pivot = dl_only.pivot(index='Model', columns='Split', values=metric)
    pivot.plot(kind='bar', ax=ax, color=['steelblue', 'coral'], edgecolor='black')
    ax.set_title(f'{metric}: LSTM vs TFT')
    ax.set_ylabel(metric); ax.set_xlabel('')
    ax.legend(title='Split')
    ax.tick_params(axis='x', rotation=0)
    if metric in ['R²', 'Bias']:
        ax.axhline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.5)
    if metric == 'WAPE':
        bar = ml_champion['overall_metrics']['test']['wape']
        ax.axhline(bar, color='red', linewidth=1, linestyle=':', alpha=0.7,
                   label=f'XGBoost test ({bar:.3f})')
        ax.legend(title='', loc='best')
plt.tight_layout()
plt.savefig('dl_overall_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


## 3. Per-item head-to-head


In [ ]:
head_to_head = []
for split in ['val', 'test']:
    lstm_split = lstm_metrics[lstm_metrics['split']==split].set_index('item_name')
    tft_split = tft_metrics[tft_metrics['split']==split].set_index('item_name')
    common = sorted(set(lstm_split.index) & set(tft_split.index))

    for metric in ['wape', 'mae', 'r2']:
        lstm_vals = lstm_split.loc[common, metric]
        tft_vals = tft_split.loc[common, metric]
        if metric == 'r2':
            lstm_wins = (lstm_vals > tft_vals).sum()
            tft_wins = (tft_vals > lstm_vals).sum()
        else:
            lstm_wins = (lstm_vals < tft_vals).sum()
            tft_wins = (tft_vals < lstm_vals).sum()
        ties = len(common) - lstm_wins - tft_wins
        head_to_head.append({
            'Split': split, 'Metric': metric,
            'LSTM wins': lstm_wins, 'TFT wins': tft_wins, 'Ties': ties,
            'LSTM win %': f"{lstm_wins / len(common) * 100:.1f}%",
        })

h2h_df = pd.DataFrame(head_to_head)
print("=" * 70)
print("PER-ITEM HEAD-TO-HEAD")
print("=" * 70)
print(h2h_df.to_string(index=False))


## 4. Per-section comparison


In [ ]:
section_compare = []
for split in ['val', 'test']:
    for model_name, pred_df in [('LSTM', lstm_pred), ('TFT', tft_pred)]:
        sub = pred_df[pred_df['split'] == split].copy()
        for section in sorted(sub['section'].dropna().unique()):
            sec = sub[sub['section'] == section]
            actual = sec['actual'].values
            predicted = sec['predicted'].values
            if len(actual) == 0 or actual.sum() == 0: continue
            mae = np.mean(np.abs(actual - predicted))
            wape = np.sum(np.abs(actual - predicted)) / np.sum(np.abs(actual))
            section_compare.append({
                'Section': section, 'Split': split, 'Model': model_name,
                'MAE': mae, 'WAPE': wape,
                'n_items': sec['item_name'].nunique(),
            })

section_df = pd.DataFrame(section_compare)
section_pivot = section_df.pivot_table(index=['Section', 'Split'], columns='Model',
                                        values='WAPE').reset_index()
section_pivot['Δ (LSTM − TFT)'] = section_pivot['LSTM'] - section_pivot['TFT']
section_pivot['Winner'] = np.where(section_pivot['LSTM'] < section_pivot['TFT'], 'LSTM', 'TFT')

print("=" * 80)
print("PER-SECTION WAPE COMPARISON")
print("=" * 80)
print(section_pivot.to_string(index=False, float_format='%.4f'))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, split in zip(axes, ['val', 'test']):
    sub = section_pivot[section_pivot['Split'] == split].copy()
    sub = sub.set_index('Section')[['LSTM', 'TFT']]
    sub.plot(kind='bar', ax=ax, color=['steelblue', 'coral'], edgecolor='black')
    ax.set_title(f'WAPE by section ({split})')
    ax.set_ylabel('WAPE'); ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=15)
    ax.legend(title='Model')
plt.tight_layout()
plt.savefig('dl_section_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


## 5. Per-volume-tier comparison


In [ ]:
df_orig = pd.read_csv(os.path.join(DATASETS_DIR, 'weekly_item_demand.csv'),
                       parse_dates=['week_start'])
item_volume = df_orig[df_orig['split']=='train'].groupby('item_name')['quantity'].mean()

def assign_tier(avg):
    if avg >= 35: return 'High (≥35/wk)'
    elif avg >= 7: return 'Medium (7-35/wk)'
    else: return 'Low (<7/wk)'

item_tiers = item_volume.apply(assign_tier).rename('volume_tier').reset_index()

tier_compare = []
for split in ['val', 'test']:
    for model_name, pred_df in [('LSTM', lstm_pred), ('TFT', tft_pred)]:
        sub = pred_df[pred_df['split'] == split].merge(item_tiers, on='item_name')
        for tier in ['High (≥35/wk)', 'Medium (7-35/wk)', 'Low (<7/wk)']:
            tier_data = sub[sub['volume_tier'] == tier]
            if len(tier_data) == 0 or tier_data['actual'].sum() == 0: continue
            actual = tier_data['actual'].values
            predicted = tier_data['predicted'].values
            tier_compare.append({
                'Tier': tier, 'Split': split, 'Model': model_name,
                'MAE': np.mean(np.abs(actual - predicted)),
                'WAPE': np.sum(np.abs(actual - predicted)) / np.sum(np.abs(actual)),
                'n_items': tier_data['item_name'].nunique(),
            })

tier_df = pd.DataFrame(tier_compare)
tier_pivot = tier_df.pivot_table(index=['Tier', 'Split'], columns='Model',
                                  values='WAPE').reset_index()
tier_pivot['Δ (LSTM − TFT)'] = tier_pivot['LSTM'] - tier_pivot['TFT']

print("=" * 80)
print("PER-VOLUME-TIER WAPE COMPARISON")
print("=" * 80)
print(tier_pivot.to_string(index=False, float_format='%.4f'))


## 6. Per-item WAPE scatter


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, split in zip(axes, ['val', 'test']):
    lstm_split = lstm_metrics[lstm_metrics['split']==split].set_index('item_name')['wape']
    tft_split = tft_metrics[tft_metrics['split']==split].set_index('item_name')['wape']
    common = sorted(set(lstm_split.index) & set(tft_split.index))
    x = lstm_split.loc[common].values
    y = tft_split.loc[common].values

    ax.scatter(x, y, alpha=0.5, s=30, edgecolor='black', linewidth=0.5)
    lim = max(x.max(), y.max()) * 1.05
    ax.plot([0, lim], [0, lim], 'r--', linewidth=1.5, label='Equal performance')
    ax.set_xlabel('LSTM WAPE'); ax.set_ylabel('TFT WAPE')
    ax.set_title(f'Per-item WAPE — {split} split')
    ax.legend()
    diffs = np.abs(x - y)
    top_diff_idx = np.argsort(diffs)[-3:]
    for idx in top_diff_idx:
        ax.annotate(common[idx][:18], (x[idx], y[idx]), fontsize=7, alpha=0.7)
plt.tight_layout()
plt.savefig('dl_per_item_scatter.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. Wilcoxon signed-rank test


In [ ]:
wilcoxon_results = []
for split in ['val', 'test']:
    lstm_split = lstm_metrics[lstm_metrics['split']==split].set_index('item_name')['wape']
    tft_split = tft_metrics[tft_metrics['split']==split].set_index('item_name')['wape']
    common = sorted(set(lstm_split.index) & set(tft_split.index))
    x = lstm_split.loc[common].values
    y = tft_split.loc[common].values
    diff = x - y
    if (diff == 0).all():
        wilcoxon_results.append({'Split': split, 'n_items': len(common),
                                  'mean_diff': 0, 'p_value': 1.0,
                                  'verdict': 'Identical predictions'})
        continue
    stat, p = stats.wilcoxon(x, y, zero_method='wilcox')
    n_lstm_wins = (diff < 0).sum()
    n_tft_wins = (diff > 0).sum()
    verdict = 'Significant' if p < 0.05 else 'Not significant'
    wilcoxon_results.append({
        'Split': split, 'n_items': len(common),
        'LSTM wins': n_lstm_wins, 'TFT wins': n_tft_wins,
        'mean_diff': diff.mean(), 'p_value': p, 'verdict': verdict,
    })

wilcoxon_df = pd.DataFrame(wilcoxon_results)
print("=" * 80)
print("WILCOXON SIGNED-RANK TEST — LSTM vs TFT (per-item WAPE)")
print("=" * 80)
print(wilcoxon_df.to_string(index=False, float_format='%.4f'))


## 8. Strategic trade-offs


In [ ]:
tradeoffs = pd.DataFrame([
    {'Criterion': 'Training time',
     'LSTM': f"~{lstm_summary.get('training_time_sec', 'n/a')}s",
     'TFT': f"~{tft_summary.get('training_time_sec', 'n/a')}s",
     'Notes': 'TFT typically slower due to attention layers'},
    {'Criterion': 'Architecture',
     'LSTM': '2-layer LSTM + MLP head',
     'TFT': 'Variable selection + LSTM + multi-head attention',
     'Notes': 'TFT is significantly more complex'},
    {'Criterion': 'Trainable parameters',
     'LSTM': '~30K (small)',
     'TFT': '~80K-120K',
     'Notes': 'TFT roughly 3-4× larger'},
    {'Criterion': 'Native uncertainty',
     'LSTM': 'No (point forecast)',
     'TFT': 'Yes (quantile loss → confidence intervals)',
     'Notes': 'TFT advantage for stock alerts (nb 08)'},
    {'Criterion': 'Variable importance',
     'LSTM': 'No native interpretability',
     'TFT': 'Native variable selection weights',
     'Notes': 'TFT advantage for soutenance defensibility'},
    {'Criterion': 'Production deployment',
     'LSTM': 'Lightweight torch.save',
     'TFT': 'Lightning checkpoint, more dependencies',
     'Notes': 'LSTM simpler in FastAPI'},
    {'Criterion': 'Suitability to dataset size',
     'LSTM': 'Good fit (small model, ~11K sequences)',
     'TFT': 'Risky (designed for larger datasets)',
     'Notes': 'TFT may overfit at this scale'},
])
print(tradeoffs.to_string(index=False))


## 9. Decision report


In [ ]:
lstm_val_wape = lstm_summary['overall_metrics']['val']['wape']
tft_val_wape = tft_summary['overall_metrics']['val']['wape']
lstm_val_r2 = lstm_summary['overall_metrics']['val']['r2']
tft_val_r2 = tft_summary['overall_metrics']['val']['r2']
lstm_val_bias = abs(lstm_summary['overall_metrics']['val']['bias'])
tft_val_bias = abs(tft_summary['overall_metrics']['val']['bias'])

lstm_test_wape = lstm_summary['overall_metrics']['test']['wape']
tft_test_wape = tft_summary['overall_metrics']['test']['wape']
ml_test_wape = ml_champion['overall_metrics']['test']['wape']

print("=" * 70)
print("DECISION REPORT — LSTM vs TFT")
print("=" * 70)
print()
print("[1] PRIMARY CRITERION — WAPE on validation")
print(f"    LSTM: {lstm_val_wape:.4f}")
print(f"    TFT:  {tft_val_wape:.4f}")
diff_pct = (max(lstm_val_wape, tft_val_wape) - min(lstm_val_wape, tft_val_wape)) / max(lstm_val_wape, tft_val_wape) * 100
print(f"    Diff: {diff_pct:+.2f}%")
primary_winner = 'LSTM' if lstm_val_wape < tft_val_wape else 'TFT'
print(f"    Winner: {primary_winner}")
print()
print("[2] TIEBREAKER 1 — R² on validation")
print(f"    LSTM: {lstm_val_r2:+.4f}")
print(f"    TFT:  {tft_val_r2:+.4f}")
r2_winner = 'LSTM' if lstm_val_r2 > tft_val_r2 else 'TFT'
print(f"    Winner: {r2_winner}")
print()
print("[3] TIEBREAKER 2 — |Bias| on validation")
print(f"    LSTM: {lstm_val_bias:.4f}")
print(f"    TFT:  {tft_val_bias:.4f}")
bias_winner = 'LSTM' if lstm_val_bias < tft_val_bias else 'TFT'
print(f"    Winner: {bias_winner}")
print()
print(f"[4] SANITY CHECK — Either model beats ML champion test WAPE ({ml_test_wape:.4f})?")
print(f"    LSTM test WAPE: {lstm_test_wape:.4f}  → {'✓ beats ML' if lstm_test_wape < ml_test_wape else '✗ does not beat ML'}")
print(f"    TFT test WAPE:  {tft_test_wape:.4f}  → {'✓ beats ML' if tft_test_wape < ml_test_wape else '✗ does not beat ML'}")
print()
print("=" * 70)
votes = [primary_winner, r2_winner, bias_winner]
lstm_votes = votes.count('LSTM')
tft_votes = votes.count('TFT')
overall = 'LSTM' if lstm_votes > tft_votes else 'TFT'
print(f"FINAL DECISION: {overall} advances to notebook 07")
print(f"Vote: LSTM {lstm_votes} – TFT {tft_votes}")
print("=" * 70)
print()
if lstm_test_wape >= ml_test_wape and tft_test_wape >= ml_test_wape:
    print("NOTE: Neither DL model beats the ML champion. This is a publishable finding —")
    print("with ~11K training sequences, deep learning has no edge over feature-rich gradient")
    print("boosting. The DL winner still advances to nb 07 for completeness, but the")
    print("final tournament is expected to confirm the ML champion as the production model.")


## 10. Save comparison summary


In [ ]:
comparison_summary = {
    'phase': 'deep_learning',
    'models_compared': ['lstm', 'tft'],
    'overall_metrics': {
        'lstm': lstm_summary['overall_metrics'],
        'tft': tft_summary['overall_metrics'],
    },
    'wilcoxon': wilcoxon_df.to_dict(orient='records'),
    'head_to_head': h2h_df.to_dict(orient='records'),
    'winner': overall,
    'votes': {'lstm': int(lstm_votes), 'tft': int(tft_votes)},
    'beats_ml_champion': {
        'lstm': bool(lstm_test_wape < ml_test_wape),
        'tft': bool(tft_test_wape < ml_test_wape),
    },
}
out_path = os.path.join(DATASETS_DIR, 'dl_comparison_summary.json')
with open(out_path, 'w') as f:
    json.dump(comparison_summary, f, indent=2, default=str)
print(f"Saved: {out_path}")
print(f"DL champion advancing to notebook 07: {overall}")
